# Exploring dataset

In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = (
    SparkSession.builder
    .appName("Spotify Exploration")
    .master("local[*]")
    .getOrCreate()
)

c:\Users\albac\OneDrive\Proyectos\spotify-analytics-hit-prediction\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [3]:
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", True)
    .option("mode", "PERMISSIVE")
    .csv("../data/raw/spotify_tracks.csv")
)

In [4]:
df.show(5)

+---+--------------------+--------------------+--------------------+--------------------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-------+--------------+-----------+
|_c0|            track_id|             artists|          album_name|          track_name|popularity|duration_ms|explicit|danceability|energy|key|loudness|mode|speechiness|acousticness|instrumentalness|liveness|valence|  tempo|time_signature|track_genre|
+---+--------------------+--------------------+--------------------+--------------------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-------+--------------+-----------+
|  0|5SuOikwiRyPMVoIQD...|         Gen Hoshino|              Comedy|              Comedy|        73|     230666|   false|       0.676| 0.461|  1|  -6.746|   0|      0.143|      0.0322|         1.01E-6|   0.358|  0.715| 87.917|            

In [5]:
df.show(5, truncate=False)

+---+----------------------+----------------------+------------------------------------------------------+--------------------------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-------+--------------+-----------+
|_c0|track_id              |artists               |album_name                                            |track_name                |popularity|duration_ms|explicit|danceability|energy|key|loudness|mode|speechiness|acousticness|instrumentalness|liveness|valence|tempo  |time_signature|track_genre|
+---+----------------------+----------------------+------------------------------------------------------+--------------------------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-------+--------------+-----------+
|0  |5SuOikwiRyPMVoIQDJUgSV|Gen Hoshino           |Comedy                                                |

In [6]:
df.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- track_id: string (nullable = true)
 |-- artists: string (nullable = true)
 |-- album_name: string (nullable = true)
 |-- track_name: string (nullable = true)
 |-- popularity: integer (nullable = true)
 |-- duration_ms: integer (nullable = true)
 |-- explicit: boolean (nullable = true)
 |-- danceability: double (nullable = true)
 |-- energy: double (nullable = true)
 |-- key: integer (nullable = true)
 |-- loudness: double (nullable = true)
 |-- mode: integer (nullable = true)
 |-- speechiness: double (nullable = true)
 |-- acousticness: double (nullable = true)
 |-- instrumentalness: double (nullable = true)
 |-- liveness: double (nullable = true)
 |-- valence: double (nullable = true)
 |-- tempo: double (nullable = true)
 |-- time_signature: integer (nullable = true)
 |-- track_genre: string (nullable = true)



In [7]:
print("Rows:", df.count())

Rows: 114000


In [8]:
print("Columns:", len(df.columns))

Columns: 21


In [9]:
print(df.columns)

['_c0', 'track_id', 'artists', 'album_name', 'track_name', 'popularity', 'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'track_genre']


In [10]:
from pyspark.sql.functions import col

In [11]:
df.filter(col("explicit") != "False").select(
    "track_id",
    "artists",
    "track_name",
    "popularity",
    "duration_ms",
    "explicit"
).show(20, truncate=False)

+----------------------+-----------------------------+----------------------------------+----------+-----------+--------+
|track_id              |artists                      |track_name                        |popularity|duration_ms|explicit|
+----------------------+-----------------------------+----------------------------------+----------+-----------+--------+
|2gRKq9rIC5i1zuxp06zJWH|Chord Overstreet             |What's Left of You                |57        |178600     |true    |
|3M6NL7PxIDZ3ArfPrBUTUP|Days N Daze                  |Flurry Rush                       |37        |169318     |true    |
|5DUAKXpyv3nL50PnAQbPS0|Get Dead                     |Fuck You                          |36        |169266     |true    |
|0njuNwIyB3DoBHC0SFUkOr|Andrew Foy;Renee Foy         |You Broke Me First                |39        |128798     |true    |
|1MGIxsVtqK2npdPmI8Dj0D|Days N Daze                  |Goodbye Lulu                      |35        |150047     |true    |
|42mkE3wbofBNickLHgdVlo|

In [12]:
df.select("explicit").distinct().show(20, truncate=False)

+--------+
|explicit|
+--------+
|true    |
|false   |
+--------+



In [13]:
# first trial i had a problem with the column "explicit", so i restarted the kernel and redid everything, which is why there is so much extra code, i will keep it for now, just in case i need to go back to it.

#### Eliminating unnecessary column

In [14]:
clean_df = df.drop("_c0")

### Examining + eliminating exact duplicates 

In [15]:
from pyspark.sql.functions import col, sum

null_counts = clean_df.select(
    [
        sum(col(c).isNull().cast("int")).alias(c)
        for c in clean_df.columns
    ]
)

null_counts.show()

+--------+-------+----------+----------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-----+--------------+-----------+
|track_id|artists|album_name|track_name|popularity|duration_ms|explicit|danceability|energy|key|loudness|mode|speechiness|acousticness|instrumentalness|liveness|valence|tempo|time_signature|track_genre|
+--------+-------+----------+----------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-----+--------------+-----------+
|       0|      1|         1|         1|         0|          0|       0|           0|     0|  0|       0|   0|          0|           0|               0|       0|      0|    0|             0|          0|
+--------+-------+----------+----------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-----+-------------

In [16]:
clean_df.filter(
    col("artists").isNull() |
    col("album_name").isNull() |
    col("track_name").isNull()
).show(truncate=False)

+----------------------+-------+----------+----------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-------+--------------+-----------+
|track_id              |artists|album_name|track_name|popularity|duration_ms|explicit|danceability|energy|key|loudness|mode|speechiness|acousticness|instrumentalness|liveness|valence|tempo  |time_signature|track_genre|
+----------------------+-------+----------+----------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-------+--------------+-----------+
|1kR4gIb7nGxHPI3D2ifs59|NULL   |NULL      |NULL      |0         |0          |false   |0.501       |0.583 |7  |-9.46   |0   |0.0605     |0.69        |0.00396         |0.0747  |0.734  |138.391|4             |k-pop      |
+----------------------+-------+----------+----------+----------+-----------+--------+------------+------+---+--------+----+

In [17]:
print("Rows after removing missing values:", clean_df.count())

Rows after removing missing values: 114000


In [18]:
clean_df.filter(
    col("artists").isNull() |
    col("album_name").isNull() |
    col("track_name").isNull()
).count()

1

In [19]:
# drops the row where artist, album_name, and track_name are all null, but keeps the rows where at least one of them is not null
clean_df = clean_df.filter(
    ~(
        col("artists").isNull() &
        col("album_name").isNull() &
        col("track_name").isNull()
    )
)

In [20]:
print("Total rows:", clean_df.count())
print("Unique track IDs:", clean_df.select("track_id").distinct().count())

Total rows: 113999
Unique track IDs: 89740


In [21]:
print(
    "Duplicate rows:",
    clean_df.count() - clean_df.dropDuplicates().count()
)

Duplicate rows: 450


In [22]:
from pyspark.sql.functions import count

duplicate_rows = (
    clean_df
    .groupBy(clean_df.columns)
    .count()
    .filter(col("count") > 1)
)

duplicate_rows.show(10, truncate=False)

+----------------------+--------------------------------------------------------------+------------------------------------------------------------+-------------------------------------------------------------------------------------------------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-------+--------------+-----------+-----+
|track_id              |artists                                                       |album_name                                                  |track_name                                                                                       |popularity|duration_ms|explicit|danceability|energy|key|loudness|mode|speechiness|acousticness|instrumentalness|liveness|valence|tempo  |time_signature|track_genre|count|
+----------------------+--------------------------------------------------------------+------------------------------------------------------------+------------------

In [23]:
print("Duplicated groups:", duplicate_rows.count())

Duplicated groups: 444


In [24]:
print(
    "Rows involved in duplicates:",
    duplicate_rows.select(
        sum(col("count"))
    ).collect()[0][0]
)

Rows involved in duplicates: 894


In [25]:
clean_df = clean_df.dropDuplicates()

In [26]:
print("Rows after removing exact duplicates:", clean_df.count())

Rows after removing exact duplicates: 113549


In [27]:
print("Unique track IDs:", clean_df.select("track_id").distinct().count())

Unique track IDs: 89740


In [28]:
track_counts = (
    clean_df
    .groupBy("track_id")
    .count()
    .orderBy(col("count").desc())
)

track_counts.show(20)

+--------------------+-----+
|            track_id|count|
+--------------------+-----+
|6S3JlDAGk3uu3NtZb...|    9|
|2Ey6v4Sekh3Z0RUSI...|    8|
|2kkvB3RNRzwjFdGha...|    8|
|61202Zzk9rk4bPMZL...|    7|
|0YLSjVxSb5FT1Bo8T...|    7|
|0RSGPiykniIg8m7Jh...|    7|
|1Gqpa08T7eBAvPQj9...|    7|
|4WJTKbNJQ41zXnb84...|    7|
|36NwMJRaCy7x77xYG...|    7|
|4XYieGKSlJlHpzB3b...|    7|
|7tbzfR8ZvZzJEzy6v...|    7|
|0e5LcankE0UyJUuCo...|    7|
|2qgXrzJsry4KgYoJC...|    7|
|4GPQDyw9hC1DiZVh0...|    7|
|2vU6bm5hVF2idVknG...|    7|
|4aqS25F3ywJ9TGnNk...|    7|
|5ZsAhuQ24mWHiduax...|    7|
|54zCdkbIALAnv8Ihi...|    7|
|5sqkarfxe7UejHTlC...|    7|
|5ftfVzSLIi5ZxYdNb...|    7|
+--------------------+-----+
only showing top 20 rows


In [29]:
print(
    "Tracks appearing more than once:",
    track_counts.filter(col("count") > 1).count()
)

Tracks appearing more than once: 16299


In [30]:
repeated_tracks = (
    track_counts
    .filter(col("count") > 1)
    .select("track_id")
)

clean_df.join(
    repeated_tracks,
    on="track_id",
    how="inner"
).select(
    "track_id",
    "track_name",
    "artists",
    "track_genre"
).orderBy("track_id").show(30, truncate=False)

+----------------------+----------------------------------------------------------------------------------------------------------+----------------------------+-----------------+
|track_id              |track_name                                                                                                |artists                     |track_genre      |
+----------------------+----------------------------------------------------------------------------------------------------------+----------------------------+-----------------+
|001APMDOl3qtx1526T11n1|Better                                                                                                    |Pink Sweat$;Kirby           |soul             |
|001APMDOl3qtx1526T11n1|Better                                                                                                    |Pink Sweat$;Kirby           |chill            |
|001YQlnDSduXd5LgBd66gT|El Tiempo Es Dinero - Remasterizado 2007                                         

### making sure all are valid values

In [31]:
features_0_1 = [
    "danceability",
    "energy",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence"
]

for feature in features_0_1:
    invalid = clean_df.filter(
        (col(feature) < 0) | (col(feature) > 1)
    ).count()

    print(f"{feature}: {invalid} invalid values")

danceability: 0 invalid values
energy: 0 invalid values
speechiness: 0 invalid values
acousticness: 0 invalid values
instrumentalness: 0 invalid values
liveness: 0 invalid values
valence: 0 invalid values


In [32]:
print(
    "Invalid popularity:",
    clean_df.filter(
        (col("popularity") < 0) | (col("popularity") > 100)
    ).count()
)

Invalid popularity: 0


In [33]:
print(
    "Invalid duration:",
    clean_df.filter(col("duration_ms") <= 0).count()
)

Invalid duration: 0


### Understanding dataset

In [34]:
clean_df.select("popularity").describe().show()

+-------+------------------+
|summary|        popularity|
+-------+------------------+
|  count|            113549|
|   mean| 33.32443262380118|
| stddev|22.283854778038975|
|    min|                 0|
|    max|               100|
+-------+------------------+



In [35]:
from pyspark.sql.functions import when

popularity_distribution = (
    clean_df
    .withColumn(
        "popularity_range",
        when(col("popularity") < 10, "0-9")
        .when(col("popularity") < 20, "10-19")
        .when(col("popularity") < 30, "20-29")
        .when(col("popularity") < 40, "30-39")
        .when(col("popularity") < 50, "40-49")
        .when(col("popularity") < 60, "50-59")
        .when(col("popularity") < 70, "60-69")
        .when(col("popularity") < 80, "70-79")
        .when(col("popularity") < 90, "80-89")
        .otherwise("90-100")
    )
    .groupBy("popularity_range")
    .count()
    .orderBy("popularity_range")
)

popularity_distribution.show()

+----------------+-----+
|popularity_range|count|
+----------------+-----+
|             0-9|22457|
|           10-19| 9279|
|           20-29|18354|
|           30-39|14910|
|           40-49|19205|
|           50-59|14532|
|           60-69| 9344|
|           70-79| 4269|
|           80-89| 1101|
|          90-100|   98|
+----------------+-----+



In [36]:
median_popularity = clean_df.approxQuantile(
    "popularity",
    [0.5],
    0.01
)[0]

print("Median popularity:", median_popularity)

Median popularity: 34.0


In [37]:
percentiles = clean_df.approxQuantile(
    "popularity",
    [0.50, 0.75, 0.80, 0.90, 0.95],
    0.01
)

for p, value in zip(
    [50, 75, 80, 90, 95],
    percentiles
):
    print(f"{p}th percentile: {value}")

    # help decide what a hit song is bc there are currently waaaaayyy more unpopular songs than popular ones. Only 1,199 songs have popularity ≥80. And only 98 have ≥90. That means if we define a hit as popularity >= 80, we'd have a very imbalanced ML problem.

50th percentile: 34.0
75th percentile: 50.0
80th percentile: 53.0
90th percentile: 62.0
95th percentile: 68.0


In [38]:
# hit distribution
from pyspark.sql.functions import when, count, col

ml_df = clean_df.withColumn(
    "hit",
    when(col("popularity") >= 53, 1).otherwise(0)
)

ml_df.groupBy("hit").count().show()

+---+-----+
|hit|count|
+---+-----+
|  1|24763|
|  0|88786|
+---+-----+



In [39]:
# percentages of hit and non-hit songs
total = ml_df.count()

ml_df.groupBy("hit").count().withColumn(
    "percentage",
    col("count") / total * 100
).show()

+---+-----+------------------+
|hit|count|        percentage|
+---+-----+------------------+
|  1|24763|21.808206148887265|
|  0|88786| 78.19179385111273|
+---+-----+------------------+



In [40]:
clean_df.select(
    "track_name",
    "artists",
    "popularity",
    "track_genre"
).orderBy(
    col("popularity").desc()
).show(20, truncate=False)

+-------------------------------------+--------------------------+----------+-----------+
|track_name                           |artists                   |popularity|track_genre|
+-------------------------------------+--------------------------+----------+-----------+
|Unholy (feat. Kim Petras)            |Sam Smith;Kim Petras      |100       |pop        |
|Unholy (feat. Kim Petras)            |Sam Smith;Kim Petras      |100       |dance      |
|Quevedo: Bzrp Music Sessions, Vol. 52|Bizarrap;Quevedo          |99        |hip-hop    |
|I'm Good (Blue)                      |David Guetta;Bebe Rexha   |98        |edm        |
|I'm Good (Blue)                      |David Guetta;Bebe Rexha   |98        |pop        |
|La Bachata                           |Manuel Turizo             |98        |reggae     |
|La Bachata                           |Manuel Turizo             |98        |latin      |
|I'm Good (Blue)                      |David Guetta;Bebe Rexha   |98        |dance      |
|La Bachat

In [41]:
from pyspark.sql.functions import avg, count

genre_popularity = (
    clean_df
    .groupBy("track_genre")
    .agg(
        avg("popularity").alias("average_popularity"),
        count("*").alias("track_count")
    )
    .orderBy(
        col("average_popularity").desc()
    )
)

genre_popularity.show(20, truncate=False)

+-----------------+------------------+-----------+
|track_genre      |average_popularity|track_count|
+-----------------+------------------+-----------+
|pop-film         |59.28028028028028 |999        |
|k-pop            |56.963927855711425|998        |
|chill            |53.7047047047047  |999        |
|sad              |52.379            |1000       |
|grunge           |49.58258258258258 |999        |
|indian           |49.528528528528525|999        |
|anime            |48.766766766766764|999        |
|emo              |48.128            |1000       |
|pop              |47.903323262839876|993        |
|sertanejo        |47.866            |1000       |
|progressive-house|46.66166166166166 |999        |
|piano            |45.27755511022044 |998        |
|mandopop         |45.025            |1000       |
|deep-house       |44.78857715430862 |998        |
|brazil           |44.668336673346694|998        |
|electronic       |44.325            |1000       |
|pagode           |44.298      

In [42]:
from pyspark.sql.functions import avg, count, col

genre_analysis = (
    ml_df
    .groupBy("track_genre")
    .agg(
        avg("popularity").alias("average_popularity"),
        count("*").alias("track_count"),
        avg("hit").alias("hit_rate")
    )
    .withColumn(
        "hit_rate_percent",
        col("hit_rate") * 100
    )
    .orderBy(
        col("hit_rate_percent").desc()
    )
)

genre_analysis.show(20, truncate=False)

+-----------------+------------------+-----------+-------------------+------------------+
|track_genre      |average_popularity|track_count|hit_rate           |hit_rate_percent  |
+-----------------+------------------+-----------+-------------------+------------------+
|pop-film         |59.28028028028028 |999        |0.9419419419419419 |94.1941941941942  |
|chill            |53.7047047047047  |999        |0.7007007007007007 |70.07007007007007 |
|pop              |47.903323262839876|993        |0.648539778449144  |64.8539778449144  |
|k-pop            |56.963927855711425|998        |0.6312625250501002 |63.126252505010015|
|hip-hop          |38.08274470232089 |991        |0.5771947527749748 |57.71947527749748 |
|grunge           |49.58258258258258 |999        |0.5685685685685685 |56.85685685685685 |
|sad              |52.379            |1000       |0.557              |55.7              |
|metal            |43.67835671342685 |998        |0.5490981963927856 |54.90981963927856 |
|british  

In [43]:
from pyspark.sql.functions import count

clean_df.select(
    count("*").alias("total_rows"),
    count("popularity").alias("popularity_non_null")
).show()

# make sure all rows have non-null popularity values and that values are correct 

+----------+-------------------+
|total_rows|popularity_non_null|
+----------+-------------------+
|    113549|             113549|
+----------+-------------------+



In [44]:
clean_df.count()

113549

### Data Cleaning

In [45]:
# Check for null or empty values in string columns
from pyspark.sql.functions import (
    col, trim, when, sum, count, avg, min, max,
    percentile_approx, first, collect_set
)

string_columns = [
    "track_id",
    "artists",
    "album_name",
    "track_name",
    "track_genre"
]

for c in string_columns:
    empty_count = clean_df.filter(
        col(c).isNull() | (trim(col(c)) == "")
    ).count()

    print(f"{c}: {empty_count} null/empty values")

track_id: 0 null/empty values
artists: 0 null/empty values
album_name: 0 null/empty values
track_name: 0 null/empty values
track_genre: 0 null/empty values


In [46]:
# Clean string columns by trimming whitespace
for c in string_columns:
    clean_df = clean_df.withColumn(
        c,
        trim(col(c))
    )

In [47]:
# Check for null or empty values in string columns after trimming
missing_track_id = clean_df.filter(
    col("track_id").isNull() | (col("track_id") == "")
).count()

print("Rows without track_id:", missing_track_id)

Rows without track_id: 0


In [48]:
# Remove rows with missing track IDs
clean_df = clean_df.filter(
    col("track_id").isNotNull() &
    (col("track_id") != "")
)

print("Rows after removing missing track IDs:", clean_df.count())

Rows after removing missing track IDs: 113549


In [49]:
# Check for null values in all columns
null_counts = clean_df.select(
    [
        sum(col(c).isNull().cast("int")).alias(c)
        for c in clean_df.columns
    ]
)

null_counts.show(truncate=False)

+--------+-------+----------+----------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-----+--------------+-----------+
|track_id|artists|album_name|track_name|popularity|duration_ms|explicit|danceability|energy|key|loudness|mode|speechiness|acousticness|instrumentalness|liveness|valence|tempo|time_signature|track_genre|
+--------+-------+----------+----------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-----+--------------+-----------+
|0       |0      |0         |0         |0         |0          |0       |0           |0     |0  |0       |0   |0          |0           |0               |0       |0      |0    |0             |0          |
+--------+-------+----------+----------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-----+-------------

In [50]:
# Check for distinct values in categorical columns
for c in ["key", "mode", "time_signature"]:
    print(f"\nValues in {c}:")
    clean_df.select(c).distinct().orderBy(c).show(20)


Values in key:
+---+
|key|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
|  5|
|  6|
|  7|
|  8|
|  9|
| 10|
| 11|
+---+


Values in mode:
+----+
|mode|
+----+
|   0|
|   1|
+----+


Values in time_signature:
+--------------+
|time_signature|
+--------------+
|             0|
|             1|
|             3|
|             4|
|             5|
+--------------+



In [51]:
# Check for invalid values in numerical columns
print(
    "Invalid key:",
    clean_df.filter(
        col("key").isNull() |
        (col("key") < -1) |
        (col("key") > 11)
    ).count()
)

print(
    "Invalid mode:",
    clean_df.filter(
        col("mode").isNull() |
        (~col("mode").isin(0, 1))
    ).count()
)

print(
    "Invalid time_signature:",
    clean_df.filter(
        col("time_signature").isNull() |
        (col("time_signature") < 1)
    ).count()
)

Invalid key: 0
Invalid mode: 0
Invalid time_signature: 163


In [52]:
clean_df.groupBy("time_signature") \
    .count() \
    .orderBy("time_signature") \
    .show()

+--------------+------+
|time_signature| count|
+--------------+------+
|             0|   163|
|             1|   967|
|             3|  9128|
|             4|101486|
|             5|  1805|
+--------------+------+



In [53]:
# Check for rows with time_signature equal to 0 because it could b incorrect data. 
clean_df.filter(
    col("time_signature") == 0
).select(
    "track_name",
    "artists",
    "time_signature"
).show(20, truncate=False)

+----------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------+
|track_name                                                                  |artists                                                                                                                                                            |time_signature|
+----------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------+
|Soothing Brown Noise Waves                                                  |Sleep Baby Sleep;White Noise Babies;White Noise For Baby Sleep;Spa Relaxation;Spa Relaxation & Spa                                                  

In [54]:
# Check for invalid tempo values
clean_df.filter(
    (col("tempo") <= 0) | (col("tempo") > 250)
).select(
    "track_id",
    "track_name",
    "tempo"
).show(20, truncate=False)

+----------------------+----------------------------------------------------------------------------+-----+
|track_id              |track_name                                                                  |tempo|
+----------------------+----------------------------------------------------------------------------+-----+
|680gfmxE9oPj43uum5USyc|Soothing Brown Noise Waves                                                  |0.0  |
|3FCjw9jRcPqw8A9k2ZkJ5p|Clean White Noise - Loopable With No Fade                                   |0.0  |
|3guM77UfyggID9e6epTkBL|Natur eigene Schlafmaschine                                                 |0.0  |
|0XsJEXpDjyrTR08Su32Vpy|Healing Noises for Insomniac                                                |0.0  |
|0q0r8JVp4dPZBbmBUsqsgO|Carmen (1997 - Remaster), Act I: Quels regardes! Quelle effronterie!        |0.0  |
|2gsRZ8eccuUBE4slJuSo5o|Rain Sleep: Deep Forest Rain                                                |0.0  |
|5ytXTFuW4KjkHw5siVJeA5|Plas

In [55]:
# Check for invalid loudness values
clean_df.select(
    "tempo",
    "loudness",
    "duration_ms"
).describe().show()

+-------+------------------+------------------+------------------+
|summary|             tempo|          loudness|       duration_ms|
+-------+------------------+------------------+------------------+
|  count|            113549|            113549|            113549|
|   mean|122.17574470052584|-8.243407991263668|228081.37084430511|
| stddev| 29.97295431392817|5.0114215655038326|106413.09852473828|
|    min|               0.0|           -49.531|              8586|
|    max|           243.372|             4.532|           5237295|
+-------+------------------+------------------+------------------+



In [56]:
# Check for tracks with unusually long durations bc 87.3 minutes is long
clean_df.orderBy(
    col("duration_ms").desc()
).select(
    "track_id",
    "track_name",
    "artists",
    "duration_ms"
).show(20, truncate=False)

+----------------------+-------------------------------------------------+------------------------------+-----------+
|track_id              |track_name                                       |artists                       |duration_ms|
+----------------------+-------------------------------------------------+------------------------------+-----------+
|3Cnz3Bu9Wcw8p3kiBTXTxp|Unity (Voyage Mix) Pt. 1                         |Tale Of Us                    |5237295    |
|0LBIf9EFHuxmuNig4JHGZo|Crossing Wires 002 - Continuous DJ Mix           |Timo Maas                     |4789026    |
|6wypxnAvmv5zYewUX3VFDd|The Lab 03 - Continuous DJ Mix Part 1            |Seth Troxler                  |4730302    |
|6JbQr97EMFD2D3Ek6gxgt1|Amnesia Ibiza Underground 10 DJ Mix              |Loco Dice                     |4563897    |
|3gFme8BdfTtFHxKmuTnSbj|House of Om - Mark Farina - Continuous Mix       |Mark Farina                   |4447520    |
|13hQAhg1owjTpTcI9xQc6c|Live In Tokyo - Continuous Mix  

In [57]:
# Check for invalid tempo values
print(
    "Invalid tempo:",
    clean_df.filter(
        col("tempo").isNull() |
        (col("tempo") <= 0)
    ).count()
)

Invalid tempo: 157


In [58]:
# Check for missing loudness values
print(
    "Missing loudness:",
    clean_df.filter(
        col("loudness").isNull()
    ).count()
)

Missing loudness: 0


### Final validation of clean_df

In [59]:
# Check for invalid values in features again, that should be between 0 and 1
features_0_1 = [
    "danceability",
    "energy",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence"
]

for feature in features_0_1:
    invalid = clean_df.filter(
        col(feature).isNull() |
        (col(feature) < 0) |
        (col(feature) > 1)
    ).count()

    print(f"{feature}: {invalid} invalid values")

danceability: 0 invalid values
energy: 0 invalid values
speechiness: 0 invalid values
acousticness: 0 invalid values
instrumentalness: 0 invalid values
liveness: 0 invalid values
valence: 0 invalid values


In [60]:
# Check for invalid popularity values again
print(
    "Invalid popularity:",
    clean_df.filter(
        col("popularity").isNull() |
        (col("popularity") < 0) |
        (col("popularity") > 100)
    ).count()
)

Invalid popularity: 0


In [61]:
# Check for invalid duration values again
print(
    "Invalid duration:",
    clean_df.filter(
        col("duration_ms").isNull() |
        (col("duration_ms") <= 0)
    ).count()
)

Invalid duration: 0


#### FINAL DATASET SUMMARY CLEAN_DF

In [62]:
print("FINAL CLEAN DATASET")
print("-------------------")

print("Total rows:", clean_df.count())

print(
    "Unique track IDs:",
    clean_df.select("track_id").distinct().count()
)

print(
    "Exact duplicate rows:",
    clean_df.count() - clean_df.dropDuplicates().count()
)

print("Number of columns:", len(clean_df.columns))

FINAL CLEAN DATASET
-------------------
Total rows: 113549
Unique track IDs: 89740
Exact duplicate rows: 0
Number of columns: 20


## Create analytics_df

In [63]:
analytics_df = clean_df

In [64]:
print("Analytics rows:", analytics_df.count())

print(
    "Unique tracks:",
    analytics_df.select("track_id").distinct().count()
)

Analytics rows: 113549
Unique tracks: 89740


## Create ml_df

In [65]:
# ML dataset shpuld contain one observation per track.
# clean_df = 113,550 rows but only 89,740 unique tracks

In [66]:
#identidy tracks with multiple genres, which could be a problem for ML models that expect one label per observation.
from pyspark.sql.functions import countDistinct

track_genre_counts = (
    clean_df
    .groupBy("track_id")
    .agg(
        countDistinct("track_genre").alias("genre_count")
    )
)

track_genre_counts.filter(
    col("genre_count") > 1
).orderBy(
    col("genre_count").desc()
).show(20)

+--------------------+-----------+
|            track_id|genre_count|
+--------------------+-----------+
|6S3JlDAGk3uu3NtZb...|          9|
|2Ey6v4Sekh3Z0RUSI...|          8|
|2kkvB3RNRzwjFdGha...|          8|
|5ftfVzSLIi5ZxYdNb...|          7|
|54zCdkbIALAnv8Ihi...|          7|
|5sqkarfxe7UejHTlC...|          7|
|5BI1XqMJK91dsEq0B...|          7|
|08kTa3SL9sV6Iy8KL...|          7|
|2aaClnypAakdAmLw7...|          7|
|6bzWr3EpSEolVwlbL...|          7|
|61202Zzk9rk4bPMZL...|          7|
|1Gqpa08T7eBAvPQj9...|          7|
|0YLSjVxSb5FT1Bo8T...|          7|
|0RSGPiykniIg8m7Jh...|          7|
|4WJTKbNJQ41zXnb84...|          7|
|36NwMJRaCy7x77xYG...|          7|
|4XYieGKSlJlHpzB3b...|          7|
|7tbzfR8ZvZzJEzy6v...|          7|
|4aqS25F3ywJ9TGnNk...|          7|
|5ZsAhuQ24mWHiduax...|          7|
+--------------------+-----------+
only showing top 20 rows


In [67]:
# check if repeated tracks have different popularity values, which could be a problem for ML models that expect one label per observation.
popularity_consistency = (
    clean_df
    .groupBy("track_id")
    .agg(
        min("popularity").alias("min_popularity"),
        max("popularity").alias("max_popularity")
    )
)

popularity_consistency.filter(
    col("min_popularity") != col("max_popularity")
).show(20)

+--------------------+--------------+--------------+
|            track_id|min_popularity|max_popularity|
+--------------------+--------------+--------------+
|1zx6GSqLYI2ynzAHn...|             0|            41|
|7vcXWUS5ML6dJ3nkv...|            39|            40|
|2PVA13ykYYHeXKlIs...|             0|             3|
|5dAMXZx98EA6x3Miq...|            40|            41|
|0Z5ok0QLLttAKsujO...|            73|            74|
|5WaXrtRow1brEASUZ...|             1|             2|
|5fWG4JDot8nHCte7g...|            66|            67|
|5d9RSSwDvlTyuwfme...|            54|            55|
|7JUUnmGyv4qi32A5c...|             0|             1|
|4346ts6RdbK3diJWl...|            41|            42|
|46musX0gdikT9TWkz...|            30|            31|
|4Am7HvT1CHDBH5bId...|             1|             2|
|2ivnsOi3RcEbskFvT...|             0|            18|
|6rbkRZKovJISNCd5c...|            44|            45|
|7GXsxXkvOvSN8qyOt...|            38|            39|
|6O3FZGHypfvAJNsY1...|            40|         

In [68]:
# number of tracks with different popularity values
print(
    "Tracks with different popularity values:",
    popularity_consistency.filter(
        col("min_popularity") != col("max_popularity")
    ).count()
)

Tracks with different popularity values: 720


In [69]:
# Check consistency of audio features
audio_features = [
    "danceability",
    "energy",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
    "loudness",
    "duration_ms",
    "key",
    "mode",
    "time_signature"
]

consistency_checks = []

for feature in audio_features:
    inconsistent = (
        clean_df
        .groupBy("track_id")
        .agg(
            min(feature).alias("min_value"),
            max(feature).alias("max_value")
        )
        .filter(
            col("min_value") != col("max_value")
        )
        .count()
    )

    consistency_checks.append((feature, inconsistent))

for feature, inconsistent in consistency_checks:
    print(f"{feature}: {inconsistent} inconsistent tracks")

danceability: 0 inconsistent tracks
energy: 0 inconsistent tracks
speechiness: 0 inconsistent tracks
acousticness: 0 inconsistent tracks
instrumentalness: 0 inconsistent tracks
liveness: 0 inconsistent tracks
valence: 0 inconsistent tracks
tempo: 0 inconsistent tracks
loudness: 0 inconsistent tracks
duration_ms: 0 inconsistent tracks
key: 0 inconsistent tracks
mode: 0 inconsistent tracks
time_signature: 0 inconsistent tracks


In [70]:
# Build ML dataset with one observation per track, aggregating audio features and popularity values.
from pyspark.sql.functions import max as spark_max

ml_df = (
    clean_df
    .groupBy("track_id")
    .agg(
        avg("popularity").alias("popularity"),

        *[
            avg(c).alias(c)
            for c in audio_features
        ],

        spark_max("explicit").alias("explicit")
    )
)

In [71]:
# check result of creating the ML dataset
print("ML rows:", ml_df.count())

print(
    "Unique track IDs:",
    ml_df.select("track_id").distinct().count()
)

print(
    "Duplicate track IDs:",
    ml_df.count() - ml_df.select("track_id").distinct().count()
)

ML rows: 89740
Unique track IDs: 89740
Duplicate track IDs: 0


In [72]:
from pyspark.sql.functions import when

popularity_distribution_ml = (
    ml_df
    .withColumn(
        "popularity_range",
        when(col("popularity") < 10, "0-9")
        .when(col("popularity") < 20, "10-19")
        .when(col("popularity") < 30, "20-29")
        .when(col("popularity") < 40, "30-39")
        .when(col("popularity") < 50, "40-49")
        .when(col("popularity") < 60, "50-59")
        .when(col("popularity") < 70, "60-69")
        .when(col("popularity") < 80, "70-79")
        .when(col("popularity") < 90, "80-89")
        .otherwise("90-100")
    )
    .groupBy("popularity_range")
    .count()
    .orderBy("popularity_range")
)

popularity_distribution_ml.show()

+----------------+-----+
|popularity_range|count|
+----------------+-----+
|             0-9|13854|
|           10-19| 9179|
|           20-29|17444|
|           30-39|13110|
|           40-49|14953|
|           50-59|11403|
|           60-69| 6676|
|           70-79| 2494|
|           80-89|  585|
|          90-100|   42|
+----------------+-----+



In [73]:
# calculating hit percentiles on ml_df bc there is 1 per track and no repetition

percentiles_ml = ml_df.approxQuantile(
    "popularity",
    [0.50, 0.75, 0.80, 0.90, 0.95],
    0.01
)

for p, value in zip(
    [50, 75, 80, 90, 95],
    percentiles_ml
):
    print(f"{p}th percentile: {value}")

    # help decide what a hit song is bc there are currently waaaaayyy more unpopular songs than popular ones. Only 585 songs have popularity ≥80. And only 42 have ≥90. That means if we define a hit as popularity >= 80, we'd have a very imbalanced ML problem.

50th percentile: 33.0
75th percentile: 48.0
80th percentile: 52.0
90th percentile: 60.0
95th percentile: 65.0


In [74]:
# hit distribution
from pyspark.sql.functions import when, count, col

ml_df = ml_df.withColumn(
    "hit",
    when(col("popularity") >= 52, 1).otherwise(0)
)

ml_df.groupBy("hit").count().show()

+---+-----+
|hit|count|
+---+-----+
|  1|18758|
|  0|70982|
+---+-----+



In [75]:
# percentages of hit and non-hit songs
total = ml_df.count()

ml_df.groupBy("hit").count().withColumn(
    "percentage",
    col("count") / total * 100
).show()

+---+-----+------------------+
|hit|count|        percentage|
+---+-----+------------------+
|  1|18758|20.902607532872743|
|  0|70982| 79.09739246712726|
+---+-----+------------------+



In [76]:
# checking class distribution again ml dataser 
total_tracks = ml_df.count()

hit_distribution = (
    ml_df
    .groupBy("hit")
    .count()
    .withColumn(
        "percentage",
        col("count") / total_tracks * 100
    )
    .orderBy("hit")
)

hit_distribution.show()

+---+-----+------------------+
|hit|count|        percentage|
+---+-----+------------------+
|  0|70982| 79.09739246712726|
|  1|18758|20.902607532872743|
+---+-----+------------------+



### Save datasets

In [77]:
processed_path = "../data/processed"

In [78]:
print(spark.sparkContext)

<SparkContext master=local[*] appName=Spotify Exploration>


In [83]:
import os

# Crear la carpeta processed si no existe
os.makedirs(processed_path, exist_ok=True)

# Convertir a Pandas y guardar como CSV
clean_pd = clean_df.toPandas()
clean_pd.to_csv(
    os.path.join(processed_path, "spotify_tracks_clean.csv"),
    index=False
)

c:\Users\albac\OneDrive\Proyectos\spotify-analytics-hit-prediction\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [84]:
#analytics_df
# Convertir a Pandas y guardar como CSV
analytics_pd = analytics_df.toPandas()
analytics_pd.to_csv(
    os.path.join(processed_path, "spotify_tracks_analytics.csv"),
    index=False
)

c:\Users\albac\OneDrive\Proyectos\spotify-analytics-hit-prediction\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [85]:
#ml_df
# Convertir a Pandas y guardar como CSV
ml_pd = ml_df.toPandas()
ml_pd.to_csv(
    os.path.join(processed_path, "spotify_tracks_ml.csv"),
    index=False
)

c:\Users\albac\OneDrive\Proyectos\spotify-analytics-hit-prediction\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
